<a href="https://colab.research.google.com/github/Yashcode007/Airfare-Prediction-Model-/blob/main/building_slm_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part1:- Our Datasets

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Step 2: Tokenize the Dataset

In this step , we will do the following:-
1) Tokenize the dataset into TokenIDs

2) Create a file called "train.bin" and "validation.bin" where we will store the tokenIDs from the entire dataset

3) We make sure the tokenIDs are stored on a disk, rather than on the RAM for efficient computations.

In [2]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

In [3]:
def process(example):
  ids = enc.encode_ordinary(example['text'])
  out = {'ids': ids, 'len': lens(ids)}
  return out

if not os.path.exists("train.bin"):
  tokenized = ds.map(
      process,
      remove_columns=['text'],
      desc = "tokenizing the splits",
      num_proc=8
  )
  #concatenate all the ids in each dataset into one large file we can use for training
  for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'], dtype=np.uint64)
    filename = f'{split}.bin'
    dtype = np.uint16
    arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
    total_batches = 1024

    idx=0
    for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
      # Batch together samples for faster write
      batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
      arr_batch = np.concatenate(batch['ids'])
      arr[idx:idx + len(arr_batch)] = arr_batch
      idx+= len(arr_batch)
    arr.flush()

NameError: name 'ds' is not defined